# 40 · Transform & semantic — dbt marts + MetricFlow, via Trino

**This is the transform tier: where raw and silver become tested, documented, curated tables.**
The lakehouse notebooks (`10`/`11`) *store and version* data; the query notebooks (`20`/`21`/`22`)
*read* it. This notebook is the layer between them that *shapes* it — **dbt** turns the Iceberg gold
into the mesh's **seven curated marts** in `iceberg.dbt.*`, each with data tests attached, and
**MetricFlow** (the dbt Semantic Layer) defines **governed metrics once** on top of those marts so
every consumer computes them the same way.

That is the thesis of the transform tier:

> **dbt turns silver into tested, documented marts. MetricFlow defines the metrics over those marts
> once, so `avg life expectancy` means the same number everywhere it is asked.**

### Where it sits

```
  BI / semantic         Lightdash, Cube (nb 41), MetricFlow metrics   <- consistent definitions
  ---------------------------------------------------------------
  transform (dbt)       staging -> marts  ->  iceberg.dbt.mart_*      <- THIS notebook
  ---------------------------------------------------------------
  lakehouse             Iceberg tables on Nessie / MinIO (nb 11)      <- the gold dbt reads
```

dbt sits **above** the lakehouse (it reads the gold, it does not re-ingest) and **below** the BI and
semantic tools that consume its marts. It materializes through **dbt-trino**: Trino executes the
model SQL and writes the result back as an Iceberg table on the Nessie `main` ref. So the marts are
just Iceberg tables — and we read them here with the **same Trino client** notebook `20` used.

> **Read-only, throughout.** Every cell is a `SELECT` / `SHOW` / `DESCRIBE` against the *already-built*
> marts. We never run `dbt` here (it is a CLI in the Dagster image, not a notebook dependency) and we
> never write to Iceberg or Nessie. The marts **are** dbt's tested output; this notebook demonstrates
> the tier by querying that output and explaining how it was produced. As in `20`/`22`, there is no
> cleanup section because nothing is created.

## Setup

The `trino` Python client is **not** in the singleuser base image (which ships `polars`, `s3fs`,
`pyarrow`, `duckdb`, `fastavro`), so we install it here — exactly as notebook `20` does. `polars`,
used to render every result frame, already ships in the image.

In [1]:
%pip install -q trino


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**, the same pattern the rest of the query wave uses. The committed
defaults are the **in-cluster** service DNS (`trino.data-mesh.svc.cluster.local:8080`); a validation
run overrides `TRINO_HOST` / `TRINO_PORT` via env **without editing the notebook**, so the notebook
never captures the resolved address — the connection is proven by what it queries back, not by
echoing the endpoint.

We pin the connection to **`catalog=iceberg`, `schema=dbt`** — the marts' home — so a bare table name
resolves there, while a fully-qualified `iceberg.dbt.<table>` still works. This Trino speaks plain
HTTP on port 8080 and takes an empty password. `q(sql)` is our tiny helper: run a statement and hand
the rows back as a **polars** DataFrame (mirroring how `20`/`22` render frames).

In [2]:
import os, trino
import polars as pl

conn = trino.dbapi.connect(
    host=os.environ.get("TRINO_HOST", "trino.data-mesh.svc.cluster.local"),
    port=int(os.environ.get("TRINO_PORT", "8080")),
    user=os.environ.get("TRINO_USER", "jupyter"),
    catalog="iceberg",
    schema="dbt",
    http_scheme="http",
)

def q(sql):
    """Run a SELECT/SHOW/DESCRIBE and return a polars DataFrame for display (house style, as in 20/22)."""
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pl.DataFrame(rows, schema=cols, orient="row")

# prove the connection by what it queries back -- the Trino version + the mart count -- not by the address
ver = q("SELECT version() AS trino_version").item(0, "trino_version")
n_marts = q("SELECT count(*) AS n FROM information_schema.tables "
            "WHERE table_schema = 'dbt' AND table_name LIKE 'mart\\_%' ESCAPE '\\'").item(0, "n")
print(f"connected (Trino {ver}) -- catalog=iceberg schema=dbt, endpoint from env")
print(f"dbt marts visible   : {n_marts}")

connected (Trino 468) -- catalog=iceberg schema=dbt, endpoint from env
dbt marts visible   : 7


## The seven marts

dbt materializes **seven marts** into `iceberg.dbt.*`, plus the `metricflow_time_spine` calendar that
the semantic layer needs. Two domains — **music** and **health** — mirror the mesh's two dataset
families. We list them from `information_schema` (never assuming a name) and print the live row count
for each; the per-mart samples below then read only what this list confirms is there.

| mart | domain | grain | what it means |
|------|--------|-------|---------------|
| `mart_artist_popularity` | music | one row per artist | Last.fm total plays + approx-distinct listeners per artist, best-effort MusicBrainz URL |
| `mart_spotify_audio` | music | one row per track | the 11 Spotify audio features + genre, rare genres (<20 tracks) dropped — the classifier's training source |
| `mart_genre_audio_profile` | music | one row per genre | mean + stddev of the 11 audio features per genre — the interpretable genre signature |
| `mart_fma_genre_tree` | music | one row per genre | the FMA genre taxonomy (id, title, parent, top-level flag) |
| `mart_country_health` | health | one row per (country, year) | 8 WHO GHO indicators pivoted wide (life expectancy, diabetes, obesity, ...) |
| `mart_state_health_trends` | health | one row per (state, year) | BRFSS chronic-condition prevalence % by US state (diabetes, asthma, COPD, depression) |
| `mart_personality_by_country` | health | one row per country | Big Five OCEAN trait means per country (>= 30 respondents) |

`metricflow_time_spine` is not a mart — it is a full daily calendar (1960-2026) that MetricFlow joins
against to build a dense, gap-aware time axis. We use it in the semantic-layer section below.

In [3]:
# discover the marts + the time spine, with live row counts (never assume the set)
objects = [r[0] for r in [tuple(x.values()) for x in q(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'dbt' ORDER BY table_name").iter_rows(named=True)]]

counts = []
for t in [o for o in objects if o.startswith("mart_") or o == "metricflow_time_spine"]:
    n = q(f"SELECT count(*) AS n FROM iceberg.dbt.{t}").item(0, "n")
    counts.append((t, "mart" if t.startswith("mart_") else "time spine", n))

pl.DataFrame(counts, schema=["object", "kind", "rows"], orient="row").sort("object")

object,kind,rows
str,str,i64
"""mart_artist_popularity""","""mart""",173993
"""mart_country_health""","""mart""",12732
"""mart_fma_genre_tree""","""mart""",164
"""mart_genre_audio_profile""","""mart""",113
"""mart_personality_by_country""","""mart""",52
"""mart_spotify_audio""","""mart""",89741
"""mart_state_health_trends""","""mart""",770
"""metricflow_time_spine""","""time spine""",24472


### Schemas — what a mart looks like

Two `DESCRIBE`s show the shape the marts land in. `mart_country_health` is the **wide-pivot** pattern
(one column per WHO indicator); `mart_state_health_trends` is the **long-to-wide** BRFSS pivot (one
column per chronic condition). Both carry `year` as an integer — which is exactly what makes them a
natural fit for MetricFlow's time axis later.

In [4]:
country_schema = q("DESCRIBE iceberg.dbt.mart_country_health").select(["Column", "Type"])
state_schema   = q("DESCRIBE iceberg.dbt.mart_state_health_trends").select(["Column", "Type"])
print("mart_country_health:")
print(country_schema)
print("\nmart_state_health_trends:")
print(state_schema)

mart_country_health:
shape: (10, 2)
┌─────────────────────────┬─────────┐
│ Column                  ┆ Type    │
│ ---                     ┆ ---     │
│ str                     ┆ str     │
╞═════════════════════════╪═════════╡
│ country                 ┆ varchar │
│ year                    ┆ integer │
│ diabetes_prevalence     ┆ double  │
│ adult_obesity           ┆ double  │
│ hypertension            ┆ double  │
│ life_expectancy         ┆ double  │
│ healthy_life_expectancy ┆ double  │
│ alcohol_consumption     ┆ double  │
│ tobacco_smoking         ┆ double  │
│ mental_health_disorders ┆ double  │
└─────────────────────────┴─────────┘

mart_state_health_trends:
shape: (6, 2)
┌────────────────┬─────────┐
│ Column         ┆ Type    │
│ ---            ┆ ---     │
│ str            ┆ str     │
╞════════════════╪═════════╡
│ state          ┆ varchar │
│ year           ┆ integer │
│ diabetes_pct   ┆ double  │
│ asthma_pct     ┆ double  │
│ copd_pct       ┆ double  │
│ depression_pct ┆ double

### Music marts — samples

Four marts over the music datasets. Each read is a small `LIMIT` (the audio and artist marts are tens
of thousands of rows) against the live table.

- **`mart_artist_popularity`** — per-artist Last.fm reach; `total_plays` summed, `n_listeners` an
  approx-distinct (HyperLogLog) count.
- **`mart_spotify_audio`** — the ML-ready feature table: 11 audio descriptors + `track_genre`, one row
  per track, the tested source for the genre classifier and Feast's `track_audio_features`.
- **`mart_genre_audio_profile`** — built *on* `mart_spotify_audio`: mean/stddev of each feature per
  genre, the interpretable baseline the classifier is compared against.
- **`mart_fma_genre_tree`** — the Free Music Archive genre taxonomy (parent/child, top-level flag).

In [5]:
# top artists by total Last.fm plays
q("""
    SELECT artist_name, total_plays, n_listeners
    FROM iceberg.dbt.mart_artist_popularity
    ORDER BY total_plays DESC
    LIMIT 5
""")

artist_name,total_plays,n_listeners
str,i64,i64
"""the beatles""",24535627,60223
"""radiohead""",22155328,63873
"""coldplay""",13448263,51415
"""pink floyd""",12849101,35355
"""metallica""",12309799,35361


In [6]:
# the audio-feature table -- one row per track, 11 descriptors + genre (showing a few)
q("""
    SELECT track_id, track_genre, danceability, energy, valence, tempo
    FROM iceberg.dbt.mart_spotify_audio
    ORDER BY track_id
    LIMIT 5
""")

track_id,track_genre,danceability,energy,valence,tempo
str,str,f64,f64,f64,f64
"""0000vdREvCVMxbQTkS888c""","""german""",0.91,0.374,0.432,104.042
"""000CC8EParg64OmTxVnZ0p""","""club""",0.269,0.516,0.341,178.174
"""000Iz0K615UepwSJ5z2RE5""","""minimal-techno""",0.686,0.56,0.108,119.997
"""000RDCYioLteXcutOjeweY""","""hip-hop""",0.679,0.77,0.839,161.721
"""000qpdoc97IMTBvF8gwcpy""","""minimal-techno""",0.519,0.431,0.234,129.971


In [7]:
# per-genre audio signature (mean of each feature), built ON mart_spotify_audio
q("""
    SELECT track_genre, n_tracks,
           round(danceability_mean, 3) AS danceability,
           round(energy_mean, 3)       AS energy,
           round(valence_mean, 3)      AS valence,
           round(tempo_mean, 1)        AS tempo
    FROM iceberg.dbt.mart_genre_audio_profile
    ORDER BY n_tracks DESC
    LIMIT 5
""")

track_genre,n_tracks,danceability,energy,valence,tempo
str,i64,f64,f64,f64,f64
"""acoustic""",1000,0.55,0.435,0.424,119.0
"""alt-rock""",999,0.535,0.754,0.518,124.7
"""afrobeat""",999,0.669,0.703,0.698,119.2
"""cantopop""",999,0.548,0.462,0.393,124.3
"""ambient""",999,0.368,0.237,0.167,111.2


In [8]:
# the FMA genre taxonomy -- the top-level (root) genres
q("""
    SELECT genre_id, genre_title, parent_id, parent_title, is_top_level
    FROM iceberg.dbt.mart_fma_genre_tree
    WHERE is_top_level = true
    ORDER BY genre_id
    LIMIT 5
""")

genre_id,genre_title,parent_id,parent_title,is_top_level
i64,str,null,null,bool
2,"""International""",null,null,true
3,"""Blues""",null,null,true
4,"""Jazz""",null,null,true
5,"""Classical""",null,null,true
8,"""Old-Time / Historic""",null,null,true


### Health marts — samples

Three marts over the health datasets. `mart_country_health` and `mart_state_health_trends` are the
**time-shaped** marts (they carry `year`) that the semantic layer builds on; `mart_personality_by_country`
is categorical (one row per country, no time axis).

- **`mart_country_health`** — 8 WHO GHO indicators pivoted to one wide row per (country, year).
- **`mart_state_health_trends`** — BRFSS chronic-condition prevalence %, one row per (US state, year),
  the tested source for Feast's `state_health_risk`.
- **`mart_personality_by_country`** — Big Five OCEAN trait means per country (>= 30 respondents).

In [9]:
# WHO country health -- most recent year, a few indicators
q("""
    SELECT country, year,
           round(life_expectancy, 1)         AS life_expectancy,
           round(healthy_life_expectancy, 1) AS healthy_life_expectancy,
           round(diabetes_prevalence, 1)     AS diabetes_prevalence
    FROM iceberg.dbt.mart_country_health
    WHERE year = 2021 AND life_expectancy IS NOT NULL
    ORDER BY life_expectancy DESC
    LIMIT 5
""")

country,year,life_expectancy,healthy_life_expectancy,diabetes_prevalence
str,i64,f64,f64,null
"""JPN""",2021,84.5,20.4,null
"""SGP""",2021,83.9,20.3,null
"""KOR""",2021,83.8,19.6,null
"""CHE""",2021,83.3,19.0,null
"""AUS""",2021,83.1,19.1,null


In [10]:
# BRFSS US-state chronic-condition prevalence -- latest year
q("""
    SELECT state, year,
           round(diabetes_pct, 1)   AS diabetes_pct,
           round(asthma_pct, 1)     AS asthma_pct,
           round(depression_pct, 1) AS depression_pct
    FROM iceberg.dbt.mart_state_health_trends
    WHERE year = 2024
    ORDER BY diabetes_pct DESC
    LIMIT 5
""")

state,year,diabetes_pct,asthma_pct,depression_pct
str,i64,f64,f64,f64
"""WV""",2024,18.4,16.7,30.2
"""VI""",2024,17.8,9.2,14.3
"""PR""",2024,17.8,16.8,18.0
"""KY""",2024,16.2,15.8,28.3
"""LA""",2024,15.4,14.0,26.6


In [11]:
# Big Five OCEAN trait means, per country, biggest respondent bases first
q("""
    SELECT country, n_respondents,
           round(extraversion, 2)       AS extraversion,
           round(openness, 2)           AS openness,
           round(conscientiousness, 2)  AS conscientiousness
    FROM iceberg.dbt.mart_personality_by_country
    ORDER BY n_respondents DESC
    LIMIT 5
""")

country,n_respondents,extraversion,openness,conscientiousness
str,i64,"decimal[38,1]","decimal[38,1]","decimal[38,1]"
"""US""",8753,3.1,3.3,3.2
"""GB""",1531,3.1,3.3,3.1
"""IN""",1464,3.2,3.2,3.1
"""AU""",974,3.1,3.3,3.1
"""CA""",924,3.1,3.4,3.1


## The modeling flow — how the marts are built

We are querying dbt's *output*. Here is how dbt produces it, so the marts above are not a black box.

### staging -> marts

This project runs a **two-layer** flow (it has no separate intermediate layer):

```
  sources          staging                       marts
  iceberg gold  ->  stg_* (ephemeral views)  ->  mart_* (materialized Iceberg tables)
  (datasets_lib)    thin clean / cast            business logic, pivots, aggregation, tests
```

- **Sources** are the Iceberg *gold* tables that `datasets_lib` lands (dbt does **not** re-ingest — it
  is the analytics-engineering layer *on top of* the gold). Declared in `models/staging/*/\_sources.yml`.
- **Staging** (`stg_brfss_prevalence`, `stg_spotify_tracks`) is a thin clean/cast over a source and is
  materialized **`ephemeral`** — it compiles *into* the mart's SQL as a CTE rather than becoming its own
  table. (Trino's Iceberg-on-Nessie catalog does not reliably support views, and `CREATE TABLE` is the
  proven write path, so staging stays inlined.)
- **Marts** (`+materialized: table`) hold the business logic — the WHO indicator pivot, the BRFSS
  break-out/response filter, the per-genre aggregation — and are written back as **Iceberg tables** in
  `iceberg.dbt` via **dbt-trino**. Those are the tables every cell above reads.

For example, `mart_state_health_trends` selects from the ephemeral `stg_brfss_prevalence`, keeps only
`break_out = 'Overall'` and `response = 'Yes'`, and pivots the four conditions to columns with
`avg(case when topic = ... )` grouped by `(state, year)`.

### Tests are part of the deliverable

Every mart carries dbt tests in its `schema.yml` — this is what "tested marts" means. `dbt build`
materializes a model **and** runs its tests; a failing test fails the build. The suite mixes three
families:

- **Uniqueness / grain** — `dbt_utils.unique_combination_of_columns: [state, year]` (and `[country, year]`)
  enforce the one-row-per-grain promise; `unique` on `track_id`, `artist_name`, `genre_id`, `country`.
- **Not-null** — on every key column (`state`, `year`, `country`, `track_genre`, `genre_title`, ...).
- **Range / domain** — `dbt_expectations.expect_column_values_to_be_between`: audio features in `[0, 1]`,
  tempo in `[0, 300]`, life expectancy in `[0, 120]`, OCEAN traits in `[1, 5]`, prevalence %s in `[0, 100]`.

The live DAG (sources -> staging -> marts, with every test node) is browsable at
**dbt-docs.weyland.lab**. dbt itself is a CLI baked into the Dagster image and orchestrated there — it
is **not** run from this notebook.

## The MetricFlow semantic layer

dbt builds the tables; **MetricFlow** (the dbt Semantic Layer) defines the **metrics** over them — once,
in the same dbt project (`models/semantic_models.yml`) — so `avg life expectancy` is computed
identically no matter who asks. A metric is declared, not re-typed into each dashboard; `mf query`
then compiles it to Trino SQL on demand.

### The two semantic models (embedded from `semantic_models.yml`)

MetricFlow is scoped here to the **time-shaped health marts** — they carry a `year`, which MetricFlow
needs as its time axis (the music marts are categorical-only and are not a natural fit).

**`country_health_sm`** over `mart_country_health`:
```yaml
entities:   ch_row (primary) = concat(country, '-', cast(year as varchar))
dimensions: country (categorical)
            obs_date (time, granularity: year) = date(concat(cast(year as varchar), '-01-01'))
measures:   avg_life_expectancy          = average(life_expectancy)
            avg_healthy_life_expectancy  = average(healthy_life_expectancy)
            avg_diabetes_prevalence      = average(diabetes_prevalence)
            countries_reporting          = count_distinct(country)
```

**`state_health_sm`** over `mart_state_health_trends`:
```yaml
dimensions: state (categorical); obs_date (time, granularity: year)
measures:   avg_diabetes_pct = average(diabetes_pct); avg_depression_pct = average(depression_pct)
            states_reporting = count_distinct(state)
```

### A couple of real metric definitions

| metric | type | measure | sits on |
|--------|------|---------|---------|
| `life_expectancy` | simple | `avg_life_expectancy` (average of `life_expectancy`) | `country_health_sm` -> `mart_country_health` |
| `diabetes_prevalence` | simple | `avg_diabetes_prevalence` | `country_health_sm` -> `mart_country_health` |
| `state_diabetes_pct` | simple | `avg_diabetes_pct` | `state_health_sm` -> `mart_state_health_trends` |
| `state_depression_pct` | simple | `avg_depression_pct` | `state_health_sm` -> `mart_state_health_trends` |

Notice the value of defining it once: `life_expectancy` the *metric* is "the average of the
`life_expectancy` column, both sexes, country-total" — that definition lives in one file, and every
tool that asks for it (a dashboard, a Cube view, a `mf query`) gets the same number.

### Demonstrating one metric against Trino

`mf` (the MetricFlow CLI) is part of the **Dagster image**, not a notebook dependency, so we do not run
it here. But we can run the **exact Trino SQL it compiles to** and see the governed number. Take the
`life_expectancy` metric grouped by its time dimension:

```bash
# in the Dagster image, NOT run here:
mf query --metrics life_expectancy --group-by metric_time__year
```

For a simple metric that is the average of a measure, grouped by the semantic model's
`agg_time_dimension` (`obs_date`, granularity `year`), MetricFlow compiles to essentially this:

In [12]:
# the compiled equivalent of `mf query --metrics life_expectancy --group-by metric_time__year`:
# obs_date is date(concat(year,'-01-01')); the metric is average(life_expectancy).
q("""
    SELECT date(concat(cast(year AS varchar), '-01-01')) AS obs_date__year,
           round(avg(life_expectancy), 2)                AS life_expectancy,
           count(*)                                      AS n_countries
    FROM iceberg.dbt.mart_country_health
    WHERE life_expectancy IS NOT NULL
      AND year BETWEEN 2010 AND 2021
    GROUP BY 1
    ORDER BY 1
""")

obs_date__year,life_expectancy,n_countries
date,f64,i64
2010-01-01,70.31,196
2011-01-01,70.72,196
2012-01-01,71.04,196
2013-01-01,71.35,196
2014-01-01,71.51,196
…,…,…
2017-01-01,72.19,196
2018-01-01,72.43,196
2019-01-01,72.64,196


The 2020-2021 dip is the pandemic showing up in the governed metric — and because the definition is
centralized, that dip reads the same in every consumer.

### Where the time spine comes in

MetricFlow does not just `GROUP BY year` — it **joins the metric to `metricflow_time_spine`** (the daily
calendar dbt materialized) and buckets the spine to the requested grain. That is what gives a metric a
**dense, gap-aware time axis**: years with no data still appear on the axis as nulls instead of silently
vanishing, and it is the machinery that makes cumulative and time-comparison metric types possible. Here
is that spine join, bucketed to year, showing the gap after 2021 explicitly:

In [13]:
# what the spine buys you: a dense year axis (from metricflow_time_spine) LEFT JOINed to the metric,
# so years with no mart data surface as nulls on the axis instead of disappearing.
q("""
    WITH spine AS (
        SELECT DISTINCT year(date_day) AS year
        FROM iceberg.dbt.metricflow_time_spine
    ),
    metric AS (
        SELECT year, avg(life_expectancy) AS life_expectancy
        FROM iceberg.dbt.mart_country_health
        WHERE life_expectancy IS NOT NULL
        GROUP BY year
    )
    SELECT s.year,
           round(m.life_expectancy, 2) AS life_expectancy
    FROM spine s
    LEFT JOIN metric m ON s.year = m.year
    WHERE s.year BETWEEN 2018 AND 2024
    ORDER BY s.year
""")

year,life_expectancy
i64,f64
2018,72.43
2019,72.64
2020,72.06
2021,71.29
2022,null
2023,null
2024,null


2022-2024 come back as nulls: the mart's WHO data ends at 2021, and the spine makes that gap **visible**
on the axis rather than dropping those years. That is precisely why MetricFlow needs the spine and why
dbt materializes it alongside the marts.

## When to reach for what

The transform tier gives you three ways to consume the same curated data — and the right one depends on
whether you need an *answer* or a *definition*.

| reach for | when… | why |
|-----------|-------|-----|
| **the marts directly** (this notebook, via Trino) | ad-hoc analysis, exploration, a one-off join, feeding a model | the marts are tested, documented Iceberg tables — query them like any other; maximum flexibility, you own the SQL |
| **MetricFlow metrics** (`mf query`, Dagster image) | a number that must be **consistent** everywhere — a KPI, a reported figure, a metric reused across dashboards | the definition lives once in `semantic_models.yml`; every consumer gets the identical calculation, spined onto a dense time axis |
| **Cube** (notebook `41`) | a semantic layer with **caching, an API, and access control** for many BI clients | Cube is the other semantic option — pre-aggregations and a served REST/SQL/GraphQL API over the same marts, where MetricFlow is dbt-native and CLI/BI-driven |

> **Query a mart when you want an answer now. Define a MetricFlow metric when the number must be the
> same everywhere. Reach for Cube (`41`) when many clients need that semantic layer served, cached, and
> access-controlled.** All three sit on the *same* dbt marts — the tested `iceberg.dbt.mart_*` tables
> this notebook read end-to-end, read-only, from inside the mesh.